# 10 — Export the review deposit (CPU only, ~2 min)

Collects every output the reviewers asked to have deposited and zips it **without model weights**:

| Folder in `deposit.zip` | Source notebook | Contents |
|---|---|---|
| `runs/<run_id>/` | 02 | `metrics.json`, `history.csv`, `preds.csv` (per-image softmax, val + test, both checkpoints) for all 224 runs |
| `audit/` | 01 | duplicate groups, leakage flags, test-set exclusion list, shortcut probes, file index |
| `compute/` | 04 | compute table |
| `gradcam/` | 05 | per-image and per-seed Grad-CAM tables |

**Attach as inputs:** the latest outputs of notebooks 02, 01, 04 and 05. Then download `/kaggle/working/deposit.zip`
from the Output tab and send it back.

In [ ]:
import os, glob, json, shutil, zipfile

OUT = '/kaggle/working/deposit'; shutil.rmtree(OUT, ignore_errors=True); os.makedirs(OUT)
KEEP_RUN_FILES = ['metrics.json', 'history.csv', 'preds.csv']

# ── notebook 02: one folder per finished run (weights skipped) ───────────────
runs = {}
for f in glob.glob('/kaggle/input/**/results/*/metrics.json', recursive=True):
    try: m = json.load(open(f))
    except Exception: continue
    if m.get('status') != 'done' or (m.get('smoke_test') and os.environ.get('ALLOW_SMOKE') != '1'): continue
    runs.setdefault(m['run_id'], os.path.dirname(f))
missing = []
for rid, src in sorted(runs.items()):
    dst = os.path.join(OUT, 'runs', rid); os.makedirs(dst, exist_ok=True)
    for fn in KEEP_RUN_FILES:
        p = os.path.join(src, fn)
        if os.path.exists(p): shutil.copy2(p, dst)
        else: missing.append(f'{rid}/{fn}')
print(f'{len(runs)} runs copied (expected 224)')
if missing: print('MISSING files:', missing[:20], '...' if len(missing) > 20 else '')

# ── notebooks 01, 04, 05: tables (csv/json/txt) and figures (png) ────────────
for sub in ['audit', 'compute', 'gradcam']:
    files = [f for f in glob.glob(f'/kaggle/input/**/{sub}/**/*', recursive=True)
             if os.path.isfile(f) and f.rsplit('.', 1)[-1].lower() in ('csv', 'json', 'txt', 'png')]
    seen = set()
    for f in files:
        rel = f.split(f'/{sub}/', 1)[1]
        if rel in seen: continue
        seen.add(rel); dst = os.path.join(OUT, sub, rel); os.makedirs(os.path.dirname(dst), exist_ok=True); shutil.copy2(f, dst)
    print(f'{sub}: {len(seen)} files' + ('' if seen else '  <-- NOT FOUND: attach this notebook output'))

In [ ]:
zp = '/kaggle/working/deposit.zip'
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for root, _, fs in os.walk(OUT):
        for fn in fs: z.write(os.path.join(root, fn), os.path.relpath(os.path.join(root, fn), OUT))
print(f'wrote {zp} ({os.path.getsize(zp) / 2**20:.1f} MB) — download it from the Output tab and send it back')